**Preflight: Assets for the Name-Search Experiments**

This notebook obtains and checks every external asset the name-search experiments depend on, before any experiment runs, as the experiment manifest (`./docs/MANIFEST.md`, §2) requires. It downloads what is missing into the git-ignored folders `./assets` (the byte-level encoder's repository and the report fonts) and `~/.cache/huggingface` (the other embedding models), starts a private PostgreSQL cluster in the git-ignored `./cache/pgdata`, and runs a small sanity check on each asset. An asset that cannot be obtained is recorded as `not_run:<reason>`, so that the later notebooks skip the configurations that need it and record them in `metrics.csv` instead of substituting another component silently. It reads no archive data and never modifies `../ocr_pipeline/extracted_text.csv`. It outputs:

1. `./results/preflight.json`: each asset's status (`ok` or `not_run:<reason>`), version or location, and the result of its sanity check
2. `./results/environment.txt`: the Python version, the platform and CPU model, and the installed packages (`pip freeze`)

**1. Configuration and Helpers**

This cell defines the paths, the asset list and the helpers that the later cells reuse. Run the notebook from `search_algorithm/` with the environment of `environment.yml`, created as described at the top of that file.

1. `ASSETS`: every asset of the manifest's §2 table, with its identifier (`E1`, `L4`, ...) and what it is used for
2. `record`: stores one asset's status, details and sanity-check result in `preflight`
3. `start_postgres` and `stop_postgres`: a private PostgreSQL cluster on a Unix socket, initialised once in `PG_DATA_DIR` with a UTF-8 locale, so that `pg_trgm` treats Hebrew letters as letters

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  shared by every later cell
#
# Each asset is tried once and recorded as `ok` or `not_run:<reason>`:
#   • downloads go to git-ignored folders, so nothing large is committed
#   • a sanity check runs on every asset that loads, using the reference
#     values the manifest quotes (e.g. cosine("Schwarzenegger", "שוורצנגר")
#     ≈ 0.84 for the byte-level encoder)
#   • a failure is caught and recorded, so one missing asset never stops
#     the preflight of the others
# ─────────────────────────────────────────────────────────────────────────────
import importlib.metadata
import importlib.util
import json
import os
import platform
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
ASSETS_DIR       = Path("assets")                       # git-ignored downloads
CACHE_DIR        = Path("cache")                        # git-ignored caches
RESULTS_DIR      = Path("results")                      # committed outputs
PREFLIGHT_FILE   = RESULTS_DIR / "preflight.json"       # each asset's status
ENVIRONMENT_FILE = RESULTS_DIR / "environment.txt"      # versions and CPU
PG_BIN           = Path(sys.executable).parent          # initdb, pg_ctl (conda env)
PG_DATA_DIR      = CACHE_DIR / "pgdata"                 # the private cluster
PG_PORT          = 5433                                 # socket file name only; no TCP
E1_REPO_URL      = "https://github.com/vedant-jumle/cross-language-phonetic-text-alignment"
E1_REPO_DIR      = ASSETS_DIR / "e1_repo"               # clone of the E1 repository
E1_CHECKPOINT    = E1_REPO_DIR / "checkpoints/best_v2/model.safetensors"
E2_REPO          = "docuracy/symphonym-v7"              # Hugging Face model id
E3A_MODEL        = "sentence-transformers/LaBSE"
E3B_MODEL        = "intfloat/multilingual-e5-small"
FONTS_DIR        = ASSETS_DIR / "fonts"                 # report fonts
RULE             = "─" * 72                             # frames every printed report

# The report's fonts (OFL / GUST licences), fetched when absent. The Liberation
# fonts come as one archive, from which only the serif files are kept.
GOOGLE_FONTS = "https://raw.githubusercontent.com/google/fonts/main/ofl"
TEX_GYRE = "https://mirrors.ctan.org/fonts/tex-gyre/opentype"
FONT_FILES = {
    "Inter.ttf": f"{GOOGLE_FONTS}/inter/Inter%5Bopsz,wght%5D.ttf",
    "texgyrepagella-regular.otf": f"{TEX_GYRE}/texgyrepagella-regular.otf",
    "texgyrepagella-bold.otf": f"{TEX_GYRE}/texgyrepagella-bold.otf",
    "texgyrepagella-italic.otf": f"{TEX_GYRE}/texgyrepagella-italic.otf",
}
LIBERATION_URL = ("https://github.com/liberationfonts/liberation-fonts/files/7261482/"
                  "liberation-fonts-ttf-2.1.5.tar.gz")
LIBERATION_KEEP = ("LiberationSerif-Regular.ttf", "LiberationSerif-Bold.ttf")

# Every asset of the manifest's §2 table, with what it serves.
ASSETS = {
    "postgresql": "PostgreSQL ≥ 16 with fuzzystrmatch and pg_trgm (L2, L3)",
    "E1":         "byte-level contrastive encoder (NumPy forward pass)",
    "E2":         "Symphonym v7 phonetic encoder",
    "E3a":        "LaBSE sentence encoder",
    "E3b":        "multilingual-e5-small sentence encoder",
    "E4":         "PyTorch, for fine-tuning E1 on archive pairs",
    "L4":         "Abydos Beider–Morse with the Hebrew-value pass",
    "anyascii":   "AnyAscii romanisation of Hebrew (L3, D4)",
    "report":     "Playwright Chromium and the report fonts",
    "qs2_llm":    "ANTHROPIC_API_KEY for the optional QS2 LLM fallback",
}

preflight: dict[str, dict] = {}


def record(asset: str, status: str, **details) -> None:
    """Store one asset's status and details in `preflight` and print a line.

    Args:
        asset: a key of ASSETS.
        status: `ok`, or `not_run:<reason>`.
        **details: version, location and sanity-check values, all JSON-ready.
    """
    preflight[asset] = {"purpose": ASSETS[asset], "status": status, **details}
    marker = "OK " if status == "ok" else "!! "
    print(f"  {marker}{asset:<11} {status}")


def short_error(error: Exception) -> str:
    """Return a one-line description of `error` for a `not_run:` status."""
    return f"{type(error).__name__}: {str(error).splitlines()[0][:120] if str(error) else ''}"


# ═════════════════════════════════════════════════════════════════════════════
# POSTGRESQL  —  a private cluster on a Unix socket, never on a TCP port
# ═════════════════════════════════════════════════════════════════════════════
def start_postgres(data_dir: Path) -> Path:
    """Initialise (once) and start a private PostgreSQL cluster.

    The cluster uses the en_US.UTF-8 locale, under which pg_trgm counts
    Hebrew letters as word characters, and listens only on a Unix socket in
    a fresh temporary folder (a short path, as macOS limits socket paths).

    Returns:
        The socket folder, to pass to psycopg as `host`.
    """
    if not (data_dir / "PG_VERSION").exists():
        data_dir.mkdir(parents=True, exist_ok=True)
        subprocess.run([PG_BIN / "initdb", "-D", data_dir, "-U", "postgres", "--auth=trust",
                        "--encoding=UTF8", "--locale=en_US.UTF-8"],
                       check=True, capture_output=True)
    stop_postgres(data_dir)   # a server left running by an interrupted run
    socket_dir = Path(tempfile.mkdtemp(prefix="pg"))
    subprocess.run([PG_BIN / "pg_ctl", "-D", data_dir, "-l", data_dir / "server.log", "-w",
                    "-o", f"-k {socket_dir} -c listen_addresses= -p {PG_PORT}", "start"],
                   check=True, capture_output=True)
    return socket_dir


def stop_postgres(data_dir: Path) -> None:
    """Stop the cluster in `data_dir` if it is running."""
    subprocess.run([PG_BIN / "pg_ctl", "-D", data_dir, "-m", "fast", "stop"],
                   check=False, capture_output=True)


RESULTS_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(exist_ok=True)
print(f"✓ Configuration and helpers loaded ({len(ASSETS)} assets to check).")

✓ Configuration and helpers loaded (10 assets to check).


**2. Environment Record**

This cell records the software and hardware the experiments run on, since the latency measurements depend on both. It writes `ENVIRONMENT_FILE` with:

1. the Python version, the operating system and the CPU model (from `sysctl` on macOS, `/proc/cpuinfo` on Linux)
2. the output of `pip freeze`, one installed package per line

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Environment record  —  versions and CPU, for reproducing the latencies
#
# `pip freeze` lists conda packages as well as pip ones, each pinned to its
# installed version, so the file doubles as a lock of the environment.
# ─────────────────────────────────────────────────────────────────────────────
if "RESULTS_DIR" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def cpu_model() -> str:
    """Return the CPU's model name, or the platform's processor string."""
    if sys.platform == "darwin":
        result = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                                capture_output=True, text=True, check=False)
        return result.stdout.strip() or platform.processor()
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.exists():
        for line in cpuinfo.read_text(encoding="utf-8").splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor()


frozen_packages = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                 capture_output=True, text=True, check=True).stdout
environment_lines = [
    f"python: {sys.version.split()[0]}",
    f"platform: {platform.platform()}",
    f"cpu: {cpu_model()}  ({os.cpu_count()} logical cores)",
    "",
    "# pip freeze",
    frozen_packages.strip(),
]
ENVIRONMENT_FILE.write_text("\n".join(environment_lines) + "\n", encoding="utf-8")

print(f"{RULE}\nEnvironment: {ENVIRONMENT_FILE}\n{RULE}")
for _line in environment_lines[:3]:
    print(f"  {_line}")
print(f"{RULE}\n  {len(frozen_packages.splitlines())} packages recorded")

────────────────────────────────────────────────────────────────────────
Environment: results/environment.txt
────────────────────────────────────────────────────────────────────────
  python: 3.14.8
  platform: macOS-26.6.2-arm64-arm-64bit-Mach-O
  cpu: Apple M1 Pro  (10 logical cores)
────────────────────────────────────────────────────────────────────────
  139 packages recorded


**3. PostgreSQL and the Lexical Methods**

This cell starts the private cluster in `PG_DATA_DIR`, installs the two extensions the lexical baselines use and checks them on known names. It also checks the two Python packages the lexical methods need: AnyAscii, which romanises Hebrew before Daitch–Mokotoff coding, and Abydos, patched as in the manifest's Appendix B so that its Beider–Morse engine runs the Hebrew-value pass. It records:

1. `postgresql`: `daitch_mokotoff('Serebrinsky')` and the trigram similarity of two Hebrew spellings, which is zero if the locale treats Hebrew letters as punctuation
2. `anyascii`: the romanisation of `רבקה`
3. `L4`: the Hebrew-value tokens shared by `סערעברינסקי` and `Serebrinsky`, which the appendix gives as `srbrinski` and `srvrinski`

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# PostgreSQL and the lexical methods  —  L2, L3, L4 and AnyAscii
#
# Abydos ships the Beider–Morse Hebrew-value rules but forces the match
# mode to `approx`; the patch fills the empty language-specific Hebrew rule
# sets and sets the private match mode, exactly as Appendix B does.
# ─────────────────────────────────────────────────────────────────────────────
if "start_postgres" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")

print(f"{RULE}\nLexical methods\n{RULE}")

# ── PostgreSQL ────────────────────────────────────────────────────────────
try:
    import psycopg

    _socket_dir = start_postgres(PG_DATA_DIR)
    try:
        with psycopg.connect(host=str(_socket_dir), port=PG_PORT, user="postgres",
                             dbname="postgres", autocommit=True) as _connection:
            _connection.execute("CREATE EXTENSION IF NOT EXISTS fuzzystrmatch")
            _connection.execute("CREATE EXTENSION IF NOT EXISTS pg_trgm")
            _version = _connection.execute("SHOW server_version").fetchone()[0]
            _dm_code = _connection.execute(
                "SELECT daitch_mokotoff('Serebrinsky')").fetchone()[0]
            _hebrew_similarity = _connection.execute(
                "SELECT similarity('רבקה', 'רבקא')").fetchone()[0]
    finally:
        stop_postgres(PG_DATA_DIR)
    if _hebrew_similarity <= 0:
        raise RuntimeError("pg_trgm finds no trigrams in Hebrew text under this locale")
    record("postgresql", "ok", version=_version, daitch_mokotoff_serebrinsky=_dm_code,
           trigram_hebrew=round(float(_hebrew_similarity), 4))
except (ImportError, OSError, subprocess.CalledProcessError, RuntimeError) as _error:
    record("postgresql", f"not_run:{short_error(_error)}")

# ── AnyAscii ──────────────────────────────────────────────────────────────
try:
    from anyascii import anyascii

    record("anyascii", "ok", romanised_rivka=anyascii("רבקה"))
except ImportError as _error:
    record("anyascii", f"not_run:{short_error(_error)}")

# ── Abydos Beider–Morse with the Hebrew-value pass ───────────────────────
try:
    import abydos.phonetic as bm_phonetic
    from abydos.phonetic._beider_morse_data import BMDATA

    _hebrew_rules = BMDATA["ash"]["hebrew"]
    for _language in BMDATA["ash"]["approx"]:
        if _language != "common" and _language not in _hebrew_rules:
            _hebrew_rules[_language] = ()
    _hebrew_values = bm_phonetic.BeiderMorse(name_mode="ash")
    _hebrew_values._match_mode = "hebrew"  # pylint: disable=protected-access
    _latin_values = bm_phonetic.BeiderMorse(
        name_mode="ash", language_arg="spanish,german,polish,russian,english")
    _latin_values._match_mode = "hebrew"  # pylint: disable=protected-access
    _shared = (set(_hebrew_values.encode("סערעברינסקי").split())
               & set(_latin_values.encode("Serebrinsky").split()))
    if not _shared:
        raise RuntimeError("the Hebrew-value pass shares no token for Serebrinsky")
    record("L4", "ok", shared_tokens_serebrinsky=sorted(_shared))
except (ImportError, KeyError, RuntimeError) as _error:
    record("L4", f"not_run:{short_error(_error)}")
print(RULE)

────────────────────────────────────────────────────────────────────────
Lexical methods
────────────────────────────────────────────────────────────────────────


  OK postgresql  ok
  OK anyascii    ok
  OK L4          ok
────────────────────────────────────────────────────────────────────────


**4. Embedding Models**

This cell obtains the four embedding models and checks each on a pair of names in two scripts. The byte-level encoder (E1) is cloned from its GitHub repository and run through the NumPy forward pass of the manifest's Appendix A, the same code that would run in production, so it needs no PyTorch. Symphonym (E2) is downloaded from Hugging Face and loaded through the `inference.py` module that ships with its weights, as its model card prescribes. LaBSE (E3a) and multilingual-e5-small (E3b) load through sentence-transformers, and E3b's inputs carry the `query: ` prefix its model card requires. It records, for each model:

1. `E1`: the cosines of `Schwarzenegger`–`שוורצנגר` and `חיים`–`Haim`, which the manifest gives as ≈ 0.84 and ≈ 0.90
2. `E2`, `E3a`, `E3b`: the cosine of `Rebeca`–`רבקה`, the model's local folder and its embedding size
3. `E4`: the PyTorch version, since E4 fine-tunes E1 and needs PyTorch only for training

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Embedding models  —  E1 (NumPy), E2 (Symphonym), E3a/E3b, and E4's PyTorch
#
# Each model is loaded, run on two spellings of one name in different
# scripts, and released. The checks confirm that the weights load and give
# cross-script similarities of the expected size, not that they are good.
# ─────────────────────────────────────────────────────────────────────────────
if "record" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


class ByteEncoder:
    """NumPy forward pass of the E1 byte-level encoder (manifest, Appendix A).

    Mirrors torch's TransformerEncoderLayer (pre-norm, ReLU) in eval mode,
    followed by mean pooling and L2 normalisation of the UTF-8 bytes.
    """

    D, H, NL = 256, 8, 6

    def __init__(self, checkpoint: Path):
        """Load the weights from the safetensors file `checkpoint`."""
        from safetensors.numpy import load_file  # pylint: disable=import-outside-toplevel
        self.weights = load_file(str(checkpoint))

    @staticmethod
    def _layer_norm(x, gain, bias, eps=1e-5):
        """Normalise the last axis of `x`, then scale by `gain` and shift by `bias`."""
        mean = x.mean(-1, keepdims=True)
        variance = ((x - mean) ** 2).mean(-1, keepdims=True)
        return (x - mean) / np.sqrt(variance + eps) * gain + bias

    def _attention(self, x: np.ndarray, prefix: str) -> np.ndarray:
        """Return the self-attention update of layer `prefix` for the states `x`."""
        w, length, head_dim = self.weights, x.shape[0], self.D // self.H
        h = self._layer_norm(x, w[prefix + "norm1.weight"], w[prefix + "norm1.bias"])
        qkv = h @ w[prefix + "self_attn.in_proj_weight"].T + w[prefix + "self_attn.in_proj_bias"]
        q, k, v = (t.reshape(length, self.H, head_dim).transpose(1, 0, 2)
                   for t in np.split(qkv, 3, -1))
        scores = q @ k.transpose(0, 2, 1) / np.sqrt(head_dim)
        scores = np.exp(scores - scores.max(-1, keepdims=True))
        scores /= scores.sum(-1, keepdims=True)
        attended = (scores @ v).transpose(1, 0, 2).reshape(length, self.D)
        out_weight = w[prefix + "self_attn.out_proj.weight"]
        return attended @ out_weight.T + w[prefix + "self_attn.out_proj.bias"]

    def _feed_forward(self, x: np.ndarray, prefix: str) -> np.ndarray:
        """Return the feed-forward update of layer `prefix` for the states `x`."""
        w = self.weights
        h = self._layer_norm(x, w[prefix + "norm2.weight"], w[prefix + "norm2.bias"])
        hidden = np.maximum(0, h @ w[prefix + "linear1.weight"].T + w[prefix + "linear1.bias"])
        return hidden @ w[prefix + "linear2.weight"].T + w[prefix + "linear2.bias"]

    def encode(self, text: str) -> np.ndarray:
        """Return the unit-length 256-dimensional embedding of `text`."""
        ids = list(text.encode("utf-8"))[:256]
        x = self.weights["embedding.weight"][ids] + self.weights["pos_embedding.weight"][:len(ids)]
        for layer in range(self.NL):
            x = x + self._attention(x, f"transformer.layers.{layer}.")
            x = x + self._feed_forward(x, f"transformer.layers.{layer}.")
        pooled = x.mean(0)
        return pooled / np.linalg.norm(pooled)

    def encode_all(self, texts: list[str]) -> np.ndarray:
        """Return the embeddings of `texts`, one row per text."""
        return np.stack([self.encode(text) for text in texts])


def load_symphonym(model_dir: Path):
    """Import Symphonym's own `inference.py` from `model_dir` and load the model.

    The v7 module defines its attention pooling with layer names and sizes
    that its own weights do not have (the weights hold `pooling.attention.0`,
    256 → 128, and `pooling.attention.3`, 128 → 1), so loading fails. Its
    `AttentionPooling` is replaced by the definition the maintainer ships
    with v8, which matches the v7 weights layer for layer.

    Returns:
        A `SymphonymModel`, whose `batch_embed` takes (text, language) pairs.
    """
    import torch  # pylint: disable=import-outside-toplevel

    class CorrectedAttentionPooling(torch.nn.Module):
        """Attention pooling as trained: Linear → Tanh → Dropout → Linear."""

        def __init__(self, hidden_dim: int, dropout: float = 0.1):
            """Build the two-layer scorer for states of size `hidden_dim`."""
            super().__init__()
            self.attention = torch.nn.Sequential(
                torch.nn.Linear(hidden_dim, hidden_dim // 2), torch.nn.Tanh(),
                torch.nn.Dropout(dropout), torch.nn.Linear(hidden_dim // 2, 1))

        def forward(self, x, mask=None):
            """Return the attention-weighted mean of `x` and the weights."""
            scores = self.attention(x).squeeze(-1)
            if mask is not None:
                scores = scores.masked_fill(~mask, float("-inf"))
            weights = torch.softmax(scores, dim=-1)
            return torch.bmm(weights.unsqueeze(1), x).squeeze(1), weights

    spec = importlib.util.spec_from_file_location("symphonym_inference",
                                                  model_dir / "inference.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.AttentionPooling = CorrectedAttentionPooling
    return module.SymphonymModel(model_dir=model_dir)


print(f"{RULE}\nEmbedding models\n{RULE}")

# ── E1: clone the repository, then run the NumPy forward pass ───────────────
try:
    if not E1_CHECKPOINT.exists():
        subprocess.run(["git", "clone", "--depth", "1", E1_REPO_URL, str(E1_REPO_DIR)],
                       check=True, capture_output=True)
    _e1 = ByteEncoder(E1_CHECKPOINT)
    _cosines = {"Schwarzenegger|שוורצנגר": float(_e1.encode("Schwarzenegger")
                                                @ _e1.encode("שוורצנגר")),
                "חיים|Haim": float(_e1.encode("חיים") @ _e1.encode("Haim"))}
    record("E1", "ok", checkpoint=str(E1_CHECKPOINT),
           cosines={pair: round(value, 3) for pair, value in _cosines.items()})
except (OSError, subprocess.CalledProcessError, KeyError) as _error:
    record("E1", f"not_run:{short_error(_error)}")

# ── E2: Symphonym from Hugging Face, through its own inference module ──────
try:
    from huggingface_hub import snapshot_download

    _e2_dir = Path(snapshot_download(E2_REPO, allow_patterns=[
        "config.json", "inference.py", "model.safetensors", "vocab/*"]))
    _e2 = load_symphonym(_e2_dir)
    _e2_vectors = _e2.batch_embed([("Rebeca", "es"), ("רבקה", "he")])
    record("E2", "ok", model_dir=str(_e2_dir), dimensions=int(_e2_vectors.shape[1]),
           cosine_rebeca=round(float(_e2_vectors[0] @ _e2_vectors[1]), 3),
           cosine_london_card_0_991=round(_e2.similarity("London", "en", "Лондон", "ru"), 3),
           note="v7 inference.py pooling replaced by the v8 definition to load the weights")
except (ImportError, OSError, RuntimeError, KeyError) as _error:
    record("E2", f"not_run:{short_error(_error)}")

# ── E3a and E3b: sentence-transformers models ────────────────────────────
for _asset, _model_name, _prefix in (("E3a", E3A_MODEL, ""), ("E3b", E3B_MODEL, "query: ")):
    try:
        from sentence_transformers import SentenceTransformer

        _vectors = SentenceTransformer(_model_name, device="cpu").encode(
            [_prefix + "Rebeca", _prefix + "רבקה"], normalize_embeddings=True)
        record(_asset, "ok", model=_model_name, dimensions=int(_vectors.shape[1]),
               cosine_rebeca=round(float(_vectors[0] @ _vectors[1]), 3))
    except (ImportError, OSError, RuntimeError) as _error:
        record(_asset, f"not_run:{short_error(_error)}")

# ── E4: fine-tuning needs PyTorch and E1's weights ──────────────────────────
try:
    if importlib.util.find_spec("torch") is None:
        raise RuntimeError("PyTorch is not installed")
    if preflight["E1"]["status"] != "ok":
        raise RuntimeError("E1 is unavailable, so there is nothing to fine-tune")
    record("E4", "ok", torch_version=importlib.metadata.version("torch"))
except RuntimeError as _error:
    record("E4", f"not_run:{short_error(_error)}")
print(RULE)

────────────────────────────────────────────────────────────────────────
Embedding models
────────────────────────────────────────────────────────────────────────
  OK E1          ok


/Users/andreiflorian/Development/cos333/cos333_gauchos_judios/search_algorithm/.conda/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 2382.68it/s]

  OK E2          ok


...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8210.53it/s]

  OK E3a         ok


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8240.20it/s]

  OK E3b         ok
  OK E4          ok
────────────────────────────────────────────────────────────────────────


**5. Report Rendering and the Optional LLM**

This cell prepares the PDF rendering of the results report and checks the one optional credential. The report keeps the design template's fonts (TeX Gyre Pagella, Inter and Liberation Serif for Hebrew), which are not installed system-wide, so they are downloaded into `FONTS_DIR` and the report loads them from there. It records:

1. `report`: whether Playwright can launch Chromium, and which of the font files are present
2. `qs2_llm`: whether `ANTHROPIC_API_KEY` is set in the environment. Without it, the QS2 query set drops the persons whose names no substitution rule applies to, and counts the drops

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Report rendering and the optional LLM  —  Chromium, fonts and the API key
#
# Fonts are fetched once; a failed download leaves the report on the CSS
# fallback stacks of the template, which the report then records as a
# deviation. The API key is only tested for presence, never printed.
# ─────────────────────────────────────────────────────────────────────────────
if "record" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def chromium_version() -> str:
    """Launch Playwright's Chromium once and return its version.

    Playwright's synchronous API refuses to run inside the notebook's own
    asyncio event loop, so callers run this function in a worker thread.
    """
    from playwright.sync_api import sync_playwright  # pylint: disable=import-outside-toplevel
    with sync_playwright() as playwright:
        browser = playwright.chromium.launch()
        version = browser.version
        browser.close()
    return version


def fetch_fonts(fonts_dir: Path) -> list[str]:
    """Download the report fonts that `fonts_dir` does not hold yet.

    Returns:
        The names of the font files present after the download.
    """
    fonts_dir.mkdir(parents=True, exist_ok=True)
    for name, url in FONT_FILES.items():
        if not (fonts_dir / name).exists():
            urllib.request.urlretrieve(url, fonts_dir / name)
    if not all((fonts_dir / name).exists() for name in LIBERATION_KEEP):
        archive = fonts_dir / "liberation.tar.gz"
        urllib.request.urlretrieve(LIBERATION_URL, archive)
        with tarfile.open(archive) as bundle:
            for member in bundle.getmembers():
                if Path(member.name).name in LIBERATION_KEEP:
                    member.name = Path(member.name).name
                    bundle.extract(member, fonts_dir, filter="data")
        archive.unlink()
    return sorted(path.name for path in fonts_dir.iterdir())


print(f"{RULE}\nReport rendering and optional credentials\n{RULE}")
try:
    from playwright.sync_api import Error as PlaywrightError

    _chromium_version = ThreadPoolExecutor(1).submit(chromium_version).result()
    _fonts = fetch_fonts(FONTS_DIR)
    _missing = [name for name in [*FONT_FILES, *LIBERATION_KEEP] if name not in _fonts]
    record("report", "ok", chromium=_chromium_version, fonts=_fonts,
           missing_fonts=_missing)
except (ImportError, OSError, PlaywrightError) as _error:
    record("report", f"not_run:{short_error(_error)}")

if os.environ.get("ANTHROPIC_API_KEY"):
    record("qs2_llm", "ok")
else:
    record("qs2_llm", "not_run:ANTHROPIC_API_KEY is not set")
print(RULE)

────────────────────────────────────────────────────────────────────────
Report rendering and optional credentials
────────────────────────────────────────────────────────────────────────


  OK report      ok
  !! qs2_llm     not_run:ANTHROPIC_API_KEY is not set
────────────────────────────────────────────────────────────────────────


**6. Consistency Check**

This cell writes `PREFLIGHT_FILE`, reads it back from disk and checks it independently of the cells above:

1. every asset of `ASSETS` has exactly one entry, and its status is `ok` or starts with `not_run:`
2. the two assets the manifest marks as required, `L4` and `anyascii`, are `ok`
3. every model recorded as `ok` has its files on disk: E1's checkpoint and E2's weights

It ends with the number of assets ready against the number checked.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  preflight.json against the asset list
#
# The file is re-read from disk, so the check covers what the later
# notebooks will see. L4 and AnyAscii are the manifest's "must run" assets:
# without them no lexical baseline beyond L1–L2 can be built.
# ─────────────────────────────────────────────────────────────────────────────
if len(preflight) < len(ASSETS):
    raise RuntimeError("Run every cell above first.")

REQUIRED_ASSETS = ("L4", "anyascii")   # the manifest's "must run" rows

PREFLIGHT_FILE.write_text(json.dumps(preflight, indent=1, ensure_ascii=False) + "\n",
                          encoding="utf-8")


def preflight_problems(path: Path) -> list[str]:
    """Return every problem found in the preflight file at `path`."""
    stored = json.loads(path.read_text(encoding="utf-8"))
    problems = [f"{asset}: missing" for asset in ASSETS if asset not in stored]
    problems += [f"{asset}: not an asset of the manifest" for asset in stored
                 if asset not in ASSETS]
    for asset, entry in stored.items():
        if not (entry["status"] == "ok" or entry["status"].startswith("not_run:")):
            problems.append(f"{asset}: status {entry['status']!r}")
    problems += [f"{asset}: required but {stored[asset]['status']}"
                 for asset in REQUIRED_ASSETS if stored.get(asset, {}).get("status") != "ok"]
    if stored.get("E1", {}).get("status") == "ok" and not E1_CHECKPOINT.exists():
        problems.append("E1: ok but its checkpoint is missing")
    if stored.get("E2", {}).get("status") == "ok" and not (
            Path(stored["E2"]["model_dir"]) / "model.safetensors").exists():
        problems.append("E2: ok but its weights are missing")
    return problems


_problems = preflight_problems(PREFLIGHT_FILE)
print(f"{RULE}\nConsistency: {PREFLIGHT_FILE}\n{RULE}")
for _asset, _entry in json.loads(PREFLIGHT_FILE.read_text(encoding="utf-8")).items():
    print(f"  {'OK ' if _entry['status'] == 'ok' else '!! '}{_asset:<11} {_entry['purpose']}")
for _problem in _problems:
    print(f"        - {_problem}")
_ready = sum(entry["status"] == "ok" for entry in preflight.values())
print(f"{RULE}\n  {_ready} of {len(ASSETS)} assets ready  "
      f"({'consistent' if not _problems else f'{len(_problems)} problem(s)'})")
print("\n✓ Preflight complete (assets obtained or recorded as not run).")
print(f"  preflight consistent: {not _problems}")

────────────────────────────────────────────────────────────────────────
Consistency: results/preflight.json
────────────────────────────────────────────────────────────────────────
  OK postgresql  PostgreSQL ≥ 16 with fuzzystrmatch and pg_trgm (L2, L3)
  OK anyascii    AnyAscii romanisation of Hebrew (L3, D4)
  OK L4          Abydos Beider–Morse with the Hebrew-value pass
  OK E1          byte-level contrastive encoder (NumPy forward pass)
  OK E2          Symphonym v7 phonetic encoder
  OK E3a         LaBSE sentence encoder
  OK E3b         multilingual-e5-small sentence encoder
  OK E4          PyTorch, for fine-tuning E1 on archive pairs
  OK report      Playwright Chromium and the report fonts
  !! qs2_llm     ANTHROPIC_API_KEY for the optional QS2 LLM fallback
────────────────────────────────────────────────────────────────────────
  9 of 10 assets ready  (consistent)

✓ Preflight complete (assets obtained or recorded as not run).
  preflight consistent: True
